In [ ]:
from pprint import pprint
import requests


port = 30000
# url = f"http://localhost:{port}/get_model_info"
# url = f"http://localhost:{port}/get_server_info"

url = f"http://localhost:{port}/generate"
data = {"text": "What is the capital of France?"}

response = requests.post(url, json=data)
pprint(response.json())

In [ ]:
import json

with open("trace/model_trace__0.jsonl", "r") as f:
    for line in f:
        print(json.loads(line).items()[0])

# print(open("trace/model_trace__0.jsonl", "r").read())
# traces = json.loads(open("trace/model_trace__0.jsonl", "r").read())   


AttributeError: 'dict_items' object has no attribute 'first'

In [3]:
import pickle
import torch

kv_inices = pickle.load(open("nonzero_counts.pkl", "rb")).cpu()
print(torch.count_nonzero(kv_inices))


tensor(6)


In [2]:
from transformers import AutoTokenizer
import os
from datasets import Dataset, Features, Value
import json
import torch

hf_root = os.environ.get("HF_ROOT", "/datasets/zhao")
cot_llama_path = os.path.join(hf_root, "models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B")
dataset_path = os.path.join(hf_root, "datasets--a-m-team--AM-DeepSeek-R1-Distilled-1.4M/am_0.9M_sample_1k.jsonl")

datasets = []
with open(dataset_path, "r") as file_:
    for i, line in enumerate(file_):
        if i >= 10: break
        datasets.append(json.loads(line))

features = Features({
    "messages": [
        {
            "role": Value("string"),
            "content": Value("string"),
            "info": {
                "source": Value("string"),
                "reference_answer": Value("string"),
                "test_case": Value("string"),
                "think_content": Value("string"),
                "answer_content": Value("string")
            }
        }
    ]
})

# Take downloading "am_0.9M_sample_1k.jsonl" as an example.

datasets = Dataset.from_list(datasets, features=features)
tokenizer = AutoTokenizer.from_pretrained(cot_llama_path)

think_tokens_list = []
answer_tokens_list = []

for example in datasets:
    assistant_message = None
    # Find the assistant message in the list of messages for the current example
    for message in example['messages']:
        if message['role'] == 'assistant':
            assistant_message = message
            break # Assuming only one assistant message per example

    if assistant_message and 'info' in assistant_message:
        think_content = assistant_message['info'].get('think_content', '') # Use .get for safety
        answer_content = assistant_message['info'].get('answer_content', '') # Use .get for safety

        # Tokenize the contents and store as PyTorch tensors
        think_tokens = tokenizer.encode(think_content, return_tensors="pt").squeeze(0)
        answer_tokens = tokenizer.encode(answer_content, return_tensors="pt").squeeze(0)
        # append a eof to the answer tokens
        answer_tokens = torch.cat([answer_tokens, torch.tensor([tokenizer.eos_token_id])])
        
        think_tokens_list.append(think_tokens)
        answer_tokens_list.append(answer_tokens)


# Store the contents to a file
trace_file_path = "./trace/am-sample.jsonl"
with open(trace_file_path, "w") as f:
    for i, (tt, at) in enumerate(zip(think_tokens_list, answer_tokens_list)):
        f.write(json.dumps({i: tt.tolist() + at.tolist()}) + "\n")
